# Lab 5.6: Task Decomposition Strategies

**What you'll build:** A multi-pass code review system -- and learn why narrow decomposition misses coverage and why per-file + cross-file analysis is the correct pattern.

**Estimated time:** 20-25 minutes

| Phase | What happens | Time |
|-------|-------------|------|
| 1 | The Wrong Way -- overly narrow decomposition misses coverage | 5 min |
| 2 | The Right Way -- per-file + cross-file analysis catches integration bugs | 10 min |
| 3 | Your Turn -- design a decomposition for a research task | 5 min |
| 4 | Stress Test -- compare prompt chaining vs dynamic decomposition | 5 min |

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

## The Setup

You are building a code review system for a project with multiple files. Each file is correct in isolation, but there are integration bugs that only appear when you look across files.

The challenge: decompose the review to catch both local and cross-file issues.

In [ ]:
# Simulated codebase with cross-file bugs
CODEBASE = {
    "api/routes.py": '''"""API endpoint definitions."""

from flask import request, jsonify

def create_user():
    """Create a new user. Expects JSON body with 'email' and 'name'."""
    data = request.json
    user = UserService.create(email=data["email"], name=data["name"])
    return jsonify({"user_id": user.id, "email": user.email}), 201

def get_user(user_id: int):
    """Get user by ID. Returns user object."""
    user = UserService.get_by_id(user_id)
    if not user:
        return jsonify({"error": "User not found"}), 404
    return jsonify({"userId": user.id, "email": user.email, "name": user.name})
''',
    "services/user_service.py": '''"""User service layer."""

class UserService:
    @staticmethod
    def create(email: str, name: str) -> "User":
        """Create user. Validates email format."""
        if "@" not in email:
            raise ValueError("Invalid email")
        return User(email=email, name=name)
    
    @staticmethod
    def get_by_id(user_id: str) -> "User":
        """Fetch user by string ID."""
        return db.query(User).filter(User.id == user_id).first()
''',
    "models/user.py": '''"""User data model."""

from dataclasses import dataclass

@dataclass
class User:
    id: int  # Auto-generated integer ID
    email: str
    name: str
    is_active: bool = True
''',
    "tests/test_api.py": '''"""API tests."""

def test_create_user():
    response = client.post("/users", json={"email": "test@test.com", "name": "Test"})
    assert response.status_code == 201
    data = response.json()
    assert "user_id" in data  # matches create_user response

def test_get_user():
    response = client.get("/users/1")
    assert response.status_code == 200
    data = response.json()
    assert "user_id" in data  # BUG: get_user returns "userId" not "user_id"
'''
}

# Ground truth: cross-file bugs
CROSS_FILE_BUGS = {
    "type_mismatch": {
        "files": ["api/routes.py", "services/user_service.py", "models/user.py"],
        "bug": "routes.py passes user_id as int, service expects str, model defines id as int. Type mismatch in service layer."
    },
    "response_key_inconsistency": {
        "files": ["api/routes.py", "tests/test_api.py"],
        "bug": "create_user returns 'user_id', get_user returns 'userId'. Test for get_user checks 'user_id' (wrong key)."
    }
}

print(f"Codebase: {len(CODEBASE)} files")
for f in CODEBASE:
    print(f"  {f}")
print(f"\nCross-file bugs: {len(CROSS_FILE_BUGS)}")
for name, bug in CROSS_FILE_BUGS.items():
    print(f"  {name}: spans {bug['files']}")

---

## Phase 1: The Wrong Approach -- Per-File Only

Review each file independently. Watch it miss the cross-file bugs.

In [ ]:
def review_single_file(filename, code):
    """Review a single file in isolation."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        messages=[{
            "role": "user",
            "content": f"""Review this code file for bugs. Report only genuine bugs, not style issues.

File: {filename}
```python
{code}
```

Return a JSON array of bugs found. Each bug: {{"location": "...", "issue": "...", "severity": "high/medium/low"}}.
If no bugs, return [].
Return ONLY the JSON array."""
        }]
    )
    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return []


print("=== PER-FILE ONLY REVIEW ===")
all_per_file_findings = []

for filename, code in CODEBASE.items():
    findings = review_single_file(filename, code)
    all_per_file_findings.extend(findings)
    print(f"\n{filename}: {len(findings)} finding(s)")
    for f in findings:
        print(f"  - {f.get('issue', '?')[:100]}")

print(f"\n=== COVERAGE CHECK ===")
found_type_mismatch = any("type" in str(f).lower() and ("int" in str(f).lower() or "str" in str(f).lower()) for f in all_per_file_findings)
found_key_inconsistency = any("userId" in str(f) or "user_id" in str(f) and "inconsisten" in str(f).lower() for f in all_per_file_findings)

print(f"Type mismatch (int vs str) detected: {found_type_mismatch}")
print(f"Response key inconsistency detected: {found_key_inconsistency}")
print()
if not found_type_mismatch or not found_key_inconsistency:
    print("MISSED cross-file bugs! Per-file review cannot catch integration issues.")
    print("Each file looks correct in isolation -- the bugs only appear when comparing files.")

---

## Phase 2: The Right Approach -- Per-File + Cross-File

Two-pass decomposition: first analyze each file deeply, then check for integration issues.

In [ ]:
# Pass 1: Per-file analysis (same as before, but we collect structured summaries)
print("=== PASS 1: Per-File Analysis ===")
file_summaries = {}

for filename, code in CODEBASE.items():
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        messages=[{
            "role": "user",
            "content": f"""Analyze this code file. Produce a structured summary for cross-file integration review.

File: {filename}
```python
{code}
```

Return a JSON object with:
- "functions": list of function/method names with their parameter types and return types
- "imports": what this file depends on
- "exports": what this file provides to other files
- "data_contracts": any JSON keys, field names, or data structures this file produces or consumes
- "local_bugs": any bugs visible within this file alone

Return ONLY the JSON object."""
        }]
    )
    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    try:
        file_summaries[filename] = json.loads(raw)
    except json.JSONDecodeError:
        file_summaries[filename] = {"raw": raw}
    
    print(f"  {filename}: summary collected")

print(f"\nCollected summaries for {len(file_summaries)} files.")

In [ ]:
# Pass 2: Cross-file integration review
print("=== PASS 2: Cross-File Integration Review ===")

cross_file_prompt = f"""You are reviewing a codebase for CROSS-FILE integration bugs.
These are bugs that are invisible when reviewing each file alone but appear when
comparing how files interact.

Per-file summaries:
{json.dumps(file_summaries, indent=2)}

Full file contents for reference:
"""

for filename, code in CODEBASE.items():
    cross_file_prompt += f"\n--- {filename} ---\n{code}\n"

cross_file_prompt += """
CHECK FOR:
1. Type mismatches: Does File A pass a type that File B expects differently?
2. Key/field name inconsistencies: Does File A produce JSON keys that File B consumes with different names?
3. API contract violations: Does the test expect a response format that differs from the actual endpoint?
4. Missing error handling: Does File A handle errors that File B can produce?

Return a JSON array of cross-file bugs. Each bug:
{"files_involved": ["..."], "bug_type": "...", "description": "...", "severity": "high/medium/low"}

Return ONLY the JSON array."""

cross_response = client.messages.create(
    model=MODEL,
    max_tokens=2048,
    messages=[{"role": "user", "content": cross_file_prompt}]
)

raw = cross_response.content[0].text.strip()
if raw.startswith("```"):
    raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
try:
    cross_file_findings = json.loads(raw)
except json.JSONDecodeError:
    cross_file_findings = []
    print(f"Parse error. Raw response: {raw[:200]}")

print(f"\nCross-file review found {len(cross_file_findings)} issue(s):\n")
for f in cross_file_findings:
    print(f"  [{f.get('severity', '?').upper()}] {f.get('bug_type', '?')}")
    print(f"  Files: {f.get('files_involved', [])}")
    print(f"  Issue: {f.get('description', '?')}")
    print()

In [ ]:
# Compare coverage
print("=" * 60)
print("COVERAGE COMPARISON")
print("=" * 60)
print(f"{'Approach':<25} {'Per-File Bugs':>15} {'Cross-File Bugs':>15}")
print(f"{'-'*25} {'-'*15} {'-'*15}")
print(f"{'Per-file only':<25} {len(all_per_file_findings):>15} {'0 (blind)':>15}")
print(f"{'Per-file + cross-file':<25} {len(all_per_file_findings):>15} {len(cross_file_findings):>15}")
print()
print("The cross-file pass catches integration bugs invisible to per-file review.")

---

## Phase 3: Your Turn

Design a decomposition strategy for a research task. The task: "Produce a comprehensive report on AI's impact on creative industries."

In [ ]:
# =============================================================
# TODO: Design the decomposition
# =============================================================
#
# Option A (Narrow -- WRONG): Pre-define 3 subcategories
# Option B (Balanced -- RIGHT): Let coordinator identify subcategories dynamically
#
# Write your decomposition plan as a dict:

narrow_plan = {
    "approach": "narrow",
    "subcategories": ["visual arts", "music production", "film production"],
    "risk": "Misses writing, gaming, advertising, fashion, architecture, and more"
}

# TODO: Design a better decomposition
balanced_plan = {
    "approach": "dynamic",
    # TODO: Define the coordinator's task
    # "coordinator_task": "First identify ALL relevant creative industry subcategories, then dispatch research subagents for each",
    # "subcategories": "determined at runtime by coordinator",
    # "synthesis": "Coordinator combines all subagent findings into unified report",
    # "advantage": "No coverage gaps -- coordinator discovers categories rather than assuming them"
}

print("Narrow plan:")
print(json.dumps(narrow_plan, indent=2))
print("\nBalanced plan:")
print(json.dumps(balanced_plan, indent=2))
print("\nTODO: Complete the balanced_plan dict.")

---

## Phase 4: Prompt Chaining vs Dynamic Decomposition

In [ ]:
print("=== WHEN TO USE EACH ===")
print()

comparison = [
    ("Invoice processing", "Prompt chaining", "Fixed steps: extract -> validate -> format. Every invoice same flow."),
    ("Debugging", "Dynamic decomposition", "Investigation path depends on what each step reveals."),
    ("Data migration", "Prompt chaining", "Fixed steps: read -> transform -> validate -> write."),
    ("Research report", "Dynamic decomposition", "Subcategories not known in advance."),
    ("Code review (multi-file)", "Per-file + cross-file", "Need both deep local and broad integration analysis."),
    ("Translation pipeline", "Prompt chaining", "Fixed: translate -> review -> format. Predictable."),
]

print(f"{'Task':<25} {'Approach':<30} {'Why'}")
print(f"{'-'*25} {'-'*30} {'-'*50}")
for task, approach, reason in comparison:
    print(f"{task:<25} {approach:<30} {reason}")

print("\nRule of thumb:")
print("  - Predictable pipeline -> Prompt chaining")
print("  - Variable investigation -> Dynamic decomposition")
print("  - Multi-file analysis -> Per-file + cross-file")

### Key Takeaways

1. **Per-file review catches local bugs; cross-file review catches integration bugs.** You need both.
2. **Narrow decomposition misses coverage.** Do not pre-define subcategories when the task scope is broad.
3. **Prompt chaining is for predictable pipelines.** Dynamic decomposition is for variable, open-ended tasks.
4. **The coordinator should discover the decomposition.** For research and analysis tasks, let the coordinator identify relevant subtasks at runtime.